# 2-Bit Input Compression Pipeline

## Status

Run the cell below to see what TFRecords and weights are already on disk. Use the output to decide which `tfrecords_exist_*` flags to set to `True` in the **Parameters** cell before running the full pipeline.

In [1]:
!./status.sh


=== TFRecords (set these flags in ### OPTIONAL TO CHANGE) ===
  [EXISTS]  tfrecords_exist_3src=True    (/nas/work/research/smartpix-box/pixelAV_datasets/shuffled/largerWindowPreliminary/dataset_3src_16x16_50x12P5_centeredIncidence_parquets/TFR_files)
  [EXISTS]  tfrecords_exist_2sc=True    (/nas/work/research/smartpix-box/pixelAV_datasets/shuffled/largerWindowPreliminary/dataset_2sc_16x16_50x12P5_centeredIncidence_parquets/TFR_files)
  [MISSING] tfrecords_exist_2su_roi=False   (/extras2/home/gdg/research/projects/smartpixels/VIAS/find_charge_cluster_centers/implementation/testbench/cocotb/release/roi_finder/sim_out/dataset_2su_-9_9_48x192_50x12P5_parquets/TFR_files)
  [MISSING] tfrecords_exist_2s_roi=False   (/extras2/home/gdg/research/projects/smartpixels/VIAS/find_charge_cluster_centers/implementation/testbench/cocotb/release/roi_finder/sim_out/dataset_2s_48x192_50x12P5_parquets/TFR_files)

=== Weights (smart-pixels/weights/) ===
  smart-pixels/weights/dataset_2s_roi_buggylabels_wei

## Clean Previous Run

⚠️ **Only run the cell below if you want to start completely from scratch.** ⚠️

It permanently deletes:
- `smart-pixels/` — all model weights and checkpoints
- `smart-pixels-ml/` — all output parquets
- `best-model/`, `npy/` — cached model copies and numpy arrays
- All `TFR_files/` directories on NAS for the training and test datasets

**Skip this cell** (do not run it) if you want to reuse existing TFRecords and trained weights from a previous run. Set `tfrecords_exist_* = True` in the Parameters cell instead.

## Setup

Edit `sys.path.insert(0, ...)` in the cell below to point to **your** local path to `two_bit_optimization_helpers`.

In [2]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "-1"

import tensorflow as tf
import random
import numpy as np
import json
import sys
sys.path.insert(0, './two_bit_optimization_helpers') # Change this line to wherever the two_bit_optimization_helpers is located 
from prepare_tfrecords import generate_tfrecords, load_tfrecords
from train import create_model, train, get_best_thresholds, cleanup_models_and_generators, save_performance_parquet
from viz_utils import viz_history
from utils import load_best_model, save_best_model, save_data_as_npy

2026-10-19 11:01:48.490026: I tensorflow/core/util/port.cc:113] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-10-19 11:01:48.513318: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-10-19 11:01:48.513341: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-10-19 11:01:48.514191: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-10-19 11:01:48.519073: I tensorflow/core/platform/cpu_feature_guar

### Parameters

Set the required variables below before running `Run All Cells`.

In [3]:
### REQUIRED TO CHANGE ##############################################################################################################################################################################
#dataset_3src_dir='/data/dajiang/smart-pixels/largerWindowPreliminary/dataset_3src_16x16_50x12P5_centeredIncidence_parquets'
#dataset_2sc_dir='/data/dajiang/smart-pixels/largerWindowPreliminary/dataset_2sc_16x16_50x12P5_centeredIncidence_parquets'
#
#weights_directory='/data/dajiang/smart-pixels/weights/dataset_3src_16x16_50x12P5_centeredIncidence_weights/'
#performance_directory_3src='/home/dajiang/smart-pixels-ml/processed_parquets/dataset_3src_16x16_50x12P5_centeredIncidence/test_dataset_3src_16x16_50x12P5_centeredIncidence/2bit_optimized/'
#performance_directory_2sc='/home/dajiang/smart-pixels-ml/processed_parquets/dataset_3src_16x16_50x12P5_centeredIncidence/test_dataset_2sc_16x16_50x12P5_centeredIncidence/2bit_optimized/'

# training dataset: 3-source, 16x16, centered incidence (non-ROI)
dataset_3src_dir='/nas/work/research/smartpix-box/pixelAV_datasets/shuffled/largerWindowPreliminary/dataset_3src_16x16_50x12P5_centeredIncidence_parquets'
# test dataset: 2-source, 16x16, centered incidence (non-ROI)
dataset_2sc_dir='/nas/work/research/smartpix-box/pixelAV_datasets/shuffled/largerWindowPreliminary/dataset_2sc_16x16_50x12P5_centeredIncidence_parquets'

# training+test dataset: 2-source, uncentered (-9 to 9), ROI geometry (48x192) — VIAS cocotb sim output (~500K rows)
dataset_2su_roi_dir='/extras2/home/gdg/research/projects/smartpixels/VIAS/fermi_roi_finder/implementation/testbench/cocotb/roi_finder/sim_out/dataset_2su_-9_9_48x192_50x12P5_parquets'
# training+test dataset: 2-source, centered incidence, ROI geometry (48x192) — VIAS cocotb sim output (~1.7M rows)
# use_roi=True selects this dataset
dataset_2s_roi_dir='/extras2/home/gdg/research/projects/smartpixels/VIAS/fermi_roi_finder/implementation/testbench/cocotb/roi_finder/sim_out/dataset_2s_48x192_50x12P5_parquets'

# output parquet: evaluated on dataset_3src validation set
performance_directory_3src='smart-pixels-ml/processed_parquets/dataset_3src_16x16_50x12P5_centeredIncidence/test_dataset_3src_16x16_50x12P5_centeredIncidence/2bit_optimized/'
# output parquet: evaluated on dataset_2sc test set
performance_directory_2sc='smart-pixels-ml/processed_parquets/dataset_3src_16x16_50x12P5_centeredIncidence/test_dataset_2sc_16x16_50x12P5_centeredIncidence/2bit_optimized/'
# output parquet: evaluated on dataset_2su_roi test set
performance_directory_2su_roi='smart-pixels-ml/processed_parquets/dataset_2su_roi/test_dataset_2su_-9_9_48x192_50x12P5_roi/2bit_optimized/'
# output parquet: evaluated on dataset_2s_roi test set with ROI-coordinate labels
performance_directory_2s_roi='smart-pixels-ml/processed_parquets/dataset_2s_roi/test_dataset_2s_48x192_50x12P5_roi/2bit_optimized/'
# output parquet: evaluated on dataset_2su_roi test set with original midplane labels
performance_directory_2su_roi_midplane='smart-pixels-ml/processed_parquets/dataset_2su_roi_midplane_labels/test_dataset_2su_-9_9_48x192_50x12P5_roi/2bit_optimized/'
# output parquet: evaluated on dataset_2s_roi test set with original midplane labels
performance_directory_2s_roi_midplane='smart-pixels-ml/processed_parquets/dataset_2s_roi_midplane_labels/test_dataset_2s_48x192_50x12P5_roi/2bit_optimized/'

# For non-quantized models: Conv2D_Max, Conv2D_Full, Conv2D_Slim, Conv1D_Full, Conv1D_Slim, Mlp_Full, Mlp_Slim
# For quantized models: QConv2D_Max, QConv2D_Full, QConv2D_Slim, QConv1D_Full, QConv1D_Slim, QMlp_Full, QMlp_Slim
model_type='QMlp_Slim'
#####################################################################################################################################################################################################

### OPTIONAL TO CHANGE ##############################################################################################################################################################################
tfrecords_exist_3src=True
tfrecords_exist_2sc=True

tfrecords_exist_2su_roi=True
# TFRecords for this dataset already exist (205 train + 52 test shards); set False to rebuild.
tfrecords_exist_2s_roi=True

initial_thresholds=[247.8, 668.4, 1662.9]
seed=42

# If you want to SKIP sections of the pipeline, change these flags to True
skip_part_1=False
skip_part_2=False

# Dataset selector:
#   True  → train and test on dataset_2s_roi (48x192 ROI, VIAS, centered incidence, ~1.7M rows)
#   False → train on dataset_3src (16x16, non-ROI), test on dataset_2sc
use_roi=True

# ROI label selector:
#   True  → use ROI-coordinate labels: x/y-coi-roi-symmetric-physical
#   False → use original labels: x/y-midplane while still using ROI inputs/filtering
use_roi_labels=True

# Which ROI dataset to use (only read when use_roi=True):
#   '2s'  → dataset_2s_roi_dir,  centered incidence
#   '2su' → dataset_2su_roi_dir, uncentered incidence (-9 to 9)
roi_dataset='2s'

# IF SKIPPING PART 1, THESE ARE REQUIRED FOR PART 2:
tfrecords_dir_train=''                                    # Path to TFRecords for train set
tfrecords_dir_val=''                                      # Path to TFRecords for val set
tfrecords_dir_test=''                                     # Path to TFRecords for test dataset
thresholds=[100, 200, 300]                                # CUSTOM THRESHOLDS
labels_scale=[]                                           # CUSTOM labels scale (Max/Full models require 4 elements, Slim models require 3 elements)
#####################################################################################################################################################################################################

### DON'T CHANGE UNLESS YOU KNOW WHAT YOU ARE DOING #################################################################################################################################################
timeslices=2
select_contained=True
train_batch_size=5000
val_batch_size=5000
threshold_offset=80.0
initial_levels=np.array([0.0, 1.0, 2.0, 3.0], dtype=np.float32)   # 2-bit in outputs (same for Part 1 and Part 2)

# epochs1/2=200 is a quick smoke-test (~5 min/part on CPU); loss is still decreasing at epoch 200
# For production results set both to 1000 (several hours on CPU)

epochs1=1000
noise1=[0,80]                                                     # Gaussian noise for Part 1
train_type1='soft_quantize_layer'                                 # soft_quantize_layer training for Part 1
soft_quantize_layer1=True                                         # soft_quantize_layer added to model in Part 1

epochs2=2000
noise2=-1                                                         # No noise for Part 2
train_type2='2bit_optimized'                                      # 2bit_optimized training for Part 2
soft_quantize_layer2=False                                        # default model architecture in Part 2
#####################################################################################################################################################################################################

# Training and test dataset parameters — controlled by use_roi flag above
if use_roi:
    # ROI: train and test on the same dataset, selected by roi_dataset above
    roi_label_suffix = '' if use_roi_labels else '_midplane_labels'
    if roi_dataset == '2su':
        roi_dir, roi_tag = dataset_2su_roi_dir, 'dataset_2su_roi'
        roi_tfrecords_exist = tfrecords_exist_2su_roi
        roi_perf_test = performance_directory_2su_roi if use_roi_labels else performance_directory_2su_roi_midplane
    elif roi_dataset == '2s':
        roi_dir, roi_tag = dataset_2s_roi_dir, 'dataset_2s_roi'
        roi_tfrecords_exist = tfrecords_exist_2s_roi
        roi_perf_test = performance_directory_2s_roi if use_roi_labels else performance_directory_2s_roi_midplane
    else:
        raise ValueError(f"roi_dataset must be '2s' or '2su', got {roi_dataset!r}")
    weights_directory               = f'smart-pixels/weights/{roi_tag}{roi_label_suffix}_weights/'
    performance_directory_train_val = f'smart-pixels-ml/processed_parquets/{roi_tag}{roi_label_suffix}/test_{roi_tag}/2bit_optimized/'
    dataset_train_src               = roi_dir
    # Midplane-label ROI TFRecords use a distinct *_midplane_labels directory and must be generated once.
    tfrecords_exist_train           = roi_tfrecords_exist if use_roi_labels else False
    select_contained_train          = False
    load_roi_train                  = True
    dataset_test_dir                = roi_dir
    tfrecords_exist_test            = roi_tfrecords_exist if use_roi_labels else False
    performance_dir_test            = roi_perf_test
    load_roi_test                   = True
    select_contained_test           = False
else:
    # Non-ROI: train on dataset_3src, test on dataset_2sc; uses flat numeric pixel columns
    weights_directory               = 'smart-pixels/weights/dataset_3src_16x16_50x12P5_centeredIncidence_weights/'
    performance_directory_train_val = performance_directory_3src
    dataset_train_src               = dataset_3src_dir
    tfrecords_exist_train           = tfrecords_exist_3src
    select_contained_train          = select_contained
    load_roi_train                  = False
    dataset_test_dir                = dataset_2sc_dir
    tfrecords_exist_test            = tfrecords_exist_2sc
    performance_dir_test            = performance_directory_2sc
    load_roi_test                   = False
    select_contained_test           = select_contained

## Part 1: Optimizing Charge Thresholds

### Set Random Seeds

In [4]:
if not skip_part_1:
    tf.random.set_seed(seed)
    random.seed(seed)

### Generate & Load TFRecords (with noise)

Converts training dataset parquet files to TFRecord format and loads them into training and validation data generators. The active dataset is `dataset_2s_roi` when `use_roi=True`, or `dataset_3src` when `use_roi=False`.

Gaussian noise (`mean=0e, stdev=80e`) is added to inputs **only in Part 1** to simulate sensor noise and make threshold optimization more robust.

In [5]:
if not skip_part_1:
    dataset_train_dir, dataset_validation_dir, tfrecords_dir_train, tfrecords_dir_val = generate_tfrecords(
        dataset_dir=dataset_train_src,
        model_type=model_type,
        train_batch_size=train_batch_size,
        val_batch_size=val_batch_size,
        select_contained=select_contained_train,
        timeslices=timeslices,
        tfrecords_exist=tfrecords_exist_train,  # True: load existing TFRecords, False: convert from parquet
        seed=seed,
        load_roi=load_roi_train,
        use_roi_labels=use_roi_labels,
    )
    print(f"TFRecords dirs train: {tfrecords_dir_train}")
    print(f"                 val: {tfrecords_dir_val}")

    training_generator1, validation_generator1 = load_tfrecords(
        tfrecords_dir_train,
        tfrecords_dir_val,
        noise=noise1,  # Gaussian noise [mean, stdev] in electrons; -1 to disable
        seed=seed,
    )
# Outputs: training_generator1, validation_generator1

Loading metadata from /extras2/home/gdg/research/projects/smartpixels/VIAS/fermi_roi_finder/implementation/testbench/cocotb/roi_finder/sim_out/dataset_2s_48x192_50x12P5_parquets/TFR_files/2t/TFR_train_slim/metadata.json
Loading metadata from /extras2/home/gdg/research/projects/smartpixels/VIAS/fermi_roi_finder/implementation/testbench/cocotb/roi_finder/sim_out/dataset_2s_48x192_50x12P5_parquets/TFR_files/2t/TFR_test_slim/metadata.json
TFRecords dirs train: /extras2/home/gdg/research/projects/smartpixels/VIAS/fermi_roi_finder/implementation/testbench/cocotb/roi_finder/sim_out/dataset_2s_48x192_50x12P5_parquets/TFR_files/2t/TFR_train_slim
                 val: /extras2/home/gdg/research/projects/smartpixels/VIAS/fermi_roi_finder/implementation/testbench/cocotb/roi_finder/sim_out/dataset_2s_48x192_50x12P5_parquets/TFR_files/2t/TFR_test_slim
Loading metadata from /extras2/home/gdg/research/projects/smartpixels/VIAS/fermi_roi_finder/implementation/testbench/cocotb/roi_finder/sim_out/dataset

Loading metadata from /extras2/home/gdg/research/projects/smartpixels/VIAS/fermi_roi_finder/implementation/testbench/cocotb/roi_finder/sim_out/dataset_2s_48x192_50x12P5_parquets/TFR_files/2t/TFR_test_slim/metadata.json


### Apply Label Scaling to Test TFRecords

Reads the label scaling factors saved during training dataset TFRecord generation and applies them when generating TFRecords for the test dataset. This ensures labels are normalized consistently between the training/validation set and the test set.

> ⚠️ **Known quirk (as of 2026-07): train and val TFRecords have slightly different `labels_scale` (~1%).**
> In the first `generate_tfrecords` call (train/val), no `labels_scale` is passed, so `OptimizedDataGenerator` computes the scale independently for each split — the 99th percentile of |labels| over the **train** parquets for `TFR_train_*` and over the **test/val** parquets for `TFR_test_*` (see `prepare_tfrecords.py`, non-`test_only` branch). E.g. for `dataset_2s_roi`: train `[128.57, 47.19, 0.529]` vs val/test `[127.47, 46.95, 0.520]`.
> Consequently the model trains on train-scale targets but is validated/evaluated on val-scale targets. Also, this cell reads the scale from **val** metadata (`tfrecords_dir_val`), and when `tfrecords_exist=True` the passed scale is ignored anyway (existing metadata wins). The effect is ~1% and applies to both ROI and non-ROI paths, so cross-run comparisons remain fair — but for strict consistency the fix is to generate train TFRecords first and pass the resulting `training_generator.labels_scale` to the val/test generation.

In [6]:
if not skip_part_1:
    with open(f'{tfrecords_dir_val}/metadata.json', 'r') as file:
        data = json.load(file)
    labels_scale = data['labels_scale']  # NOTE: read from VAL metadata, which can differ ~1% from train scale (see markdown above)

    _, tfrecords_dir_test = generate_tfrecords(
        dataset_dir=dataset_test_dir,
        train_batch_size=train_batch_size, 
        val_batch_size=val_batch_size,
        select_contained=select_contained_test,
        timeslices=timeslices, 
        seed=seed, 
        max_workers=1, 
        tfrecords_exist=tfrecords_exist_test,
        model_type=model_type,
        labels_scale=np.array(labels_scale),    # ignored when tfrecords_exist=True (existing metadata wins)
        test_only=True,                         # only generate test TFRecords, no train/val generators
        load_roi=load_roi_test,
        use_roi_labels=use_roi_labels,
    )
    print(f"TFRecords dirs test: {tfrecords_dir_test}")
# Outputs: tfrecords_dir_test

Loading metadata from /extras2/home/gdg/research/projects/smartpixels/VIAS/fermi_roi_finder/implementation/testbench/cocotb/roi_finder/sim_out/dataset_2s_48x192_50x12P5_parquets/TFR_files/2t/TFR_test_slim/metadata.json
TFRecords dirs test: /extras2/home/gdg/research/projects/smartpixels/VIAS/fermi_roi_finder/implementation/testbench/cocotb/roi_finder/sim_out/dataset_2s_48x192_50x12P5_parquets/TFR_files/2t/TFR_test_slim


### Train Model (Soft Quantize Layer)

Build and train a model with a **soft quantize layer** appended: a differentiable approximation of the digitization step that allows the 3 charge thresholds to be learned via backpropagation.

The base model architecture is set by `model_type`.

The model weights are discarded after this step: only the learned thresholds matter.

In [7]:
if not skip_part_1:
    model1 = create_model(
        model_type=model_type,
        timeslices=timeslices,
        soft_quantize_layer=soft_quantize_layer1,  # True: append soft quantize layer for threshold learning
        initial_thresholds=initial_thresholds,      # starting guess for the 3 charge boundaries (in electrons)
        threshold_offset=threshold_offset,          # offset applied to thresholds during initialization
        initial_levels=initial_levels,              # 2-bit output levels [0, 1, 2, 3]
    )
    
    checkpoints_directory1, fingerprint1, history1 = train(
        model=model1,
        model_type=model_type, 
        weights_directory=weights_directory,
        training_generator=training_generator1,
        validation_generator=validation_generator1, 
        timeslices=timeslices,
        train_type=train_type1,  # 'soft_quantize_layer': uses cosine annealing on the quantizer sharpness
        epochs=epochs1,          # up to 1000 epochs; all checkpoints saved (not just best)
        seed=seed, 
    )
    print(f"Epochs completed: {len(history1.history['loss'])}")
    print(f"Best val loss:    {min(history1.history['val_loss']):.4f}")
# Outputs: checkpoints_directory1, fingerprint1, history1

2026-10-19 11:01:50.606013: E external/local_xla/xla/stream_executor/cuda/cuda_driver.cc:274] failed call to cuInit: CUDA_ERROR_NO_DEVICE: no CUDA-capable device is detected
2026-10-19 11:01:50.606030: I external/local_xla/xla/stream_executor/cuda/cuda_diagnostics.cc:129] retrieving CUDA diagnostic information for host: thorin
2026-10-19 11:01:50.606033: I external/local_xla/xla/stream_executor/cuda/cuda_diagnostics.cc:136] hostname: thorin
2026-10-19 11:01:50.606098: I external/local_xla/xla/stream_executor/cuda/cuda_diagnostics.cc:159] libcuda reported version is: 535.309.1
2026-10-19 11:01:50.606107: I external/local_xla/xla/stream_executor/cuda/cuda_diagnostics.cc:163] kernel reported version is: 535.309.1
2026-10-19 11:01:50.606109: I external/local_xla/xla/stream_executor/cuda/cuda_diagnostics.cc:241] kernel version seems to match DSO: 535.309.1


Model: "smrtpxl_regression"
__________________________________________________________________________________________________
 Layer (type)                Output Shape                 Param #   Connected to                  
 input_pxls (InputLayer)     [(None, 16, 16, 2)]          0         []                            
                                                                                                  
 soft_quantizer_output (Sof  (None, 16, 16, 2)            4         ['input_pxls[0][0]']          
 tQuantizeLayer)                                                                                  
                                                                                                  
 average_pooling2d (Average  (None, 16, 1, 2)             0         ['soft_quantizer_output[0][0]'
 Pooling2D)                                                         ]                             
                                                                                 

KeyboardInterrupt: 

### Extract Thresholds & Digitize Inputs

Scans all checkpoints from **Part 1**, loads the one with the lowest validation loss, and extracts the learned charge thresholds and levels from the soft quantize layer.

These thresholds define how continuous charge values are mapped to 2-bit integers (0–3) in **Part 2**. The model itself is discarded.

In [ ]:
if not skip_part_1:
    thresholds, levels = get_best_thresholds(
        checkpoints=checkpoints_directory1,  # scans all saved checkpoints, picks lowest val_loss
        model_type=model_type,
        timeslices=timeslices,
        initial_thresholds=initial_thresholds,
        threshold_offset=threshold_offset,
        initial_levels=initial_levels,
    )
# Outputs: thresholds, levels

### Cleanup

Deletes **Part 1** model and data generators, clears the TF session, and forces garbage collection to free memory before **Part 2**.

In [ ]:
if not skip_part_1:
    cleanup_models_and_generators([model1, training_generator1, validation_generator1])

## Part 2: Training on Optimized Thresholds

Trains the final model on inputs that have been hard-digitized to 2-bit using the thresholds from **Part 1**. Unlike **Part 1**, there is no soft quantize layer and no noise: the model learns directly on the discrete integer inputs it will see in hardware.

### Load & Digitize TFRecords (2-bit)

Reloads the same training dataset TFRecords used in **Part 1** (`dataset_2s_roi` or `dataset_3src` depending on `use_roi`), but this time **without noise** and with inputs **digitized to 2-bit** using the thresholds learned in **Part 1**.

Each pixel charge value is mapped to a discrete level in `[0, 1, 2, 3]` before being fed to the model — simulating what the actual hardware will do.

In [ ]:
if not skip_part_2:
    training_generator2, validation_generator2 = load_tfrecords(
        tfrecords_dir_train, 
        tfrecords_dir_val,
        noise=noise2,                      # -1: no noise (unlike Part 1)
        digitize=True,                     # apply hard digitization to inputs
        digitize_levels=initial_levels,    # 2-bit output levels [0, 1, 2, 3]
        digitize_thresholds=thresholds,    # thresholds learned in Part 1
        seed=seed,
    )
    print(f"Digitization thresholds: {thresholds}")
# Outputs: training_generator2, validation_generator2

### Reset Random Seeds

Resets TensorFlow and Python random seeds to ensure reproducible weight initialization for the **Part 2** model.

In [ ]:
if not skip_part_2:
    tf.random.set_seed(seed)
    random.seed(seed)

### Train Model (2-bit Optimized)

Builds and trains the **final model** on hard-digitized 2-bit inputs. No soft quantize layer: this is the standard model architecture that will be used for inference.

In [ ]:
if not skip_part_2:
    model2 = create_model(
        model_type=model_type,
        timeslices=timeslices,
        soft_quantize_layer=soft_quantize_layer2,  # False: standard architecture, no quantize layer
    )
    
    checkpoints_directory2, fingerprint2, history2 = train(
        model=model2,
        model_type=model_type, 
        weights_directory=weights_directory,
        training_generator=training_generator2,
        validation_generator=validation_generator2, 
        timeslices=timeslices,
        train_type=train_type2,  # '2bit_optimized': standard training, no annealing scheduler
        epochs=epochs2,          # up to 1000 epochs; all checkpoints saved (not just best)
        seed=seed, 
    )
    print(f"Epochs completed: {len(history2.history['loss'])}")
    print(f"Best val loss:    {min(history2.history['val_loss']):.4f}")
# Outputs: checkpoints_directory2, fingerprint2, history2

### Evaluate on Training Dataset & Save Results

Loads the best **Part 2** checkpoint (lowest validation loss), runs predictions on the validation set, and saves results to a parquet file for downstream analysis.

Output directory is `performance_directory_train_val` — set to `dataset_2s_roi` when `use_roi=True`, or `dataset_3src` when `use_roi=False`.

In [ ]:
if not skip_part_2:
    save_performance_parquet(
        checkpoints=checkpoints_directory2,           # scans all checkpoints, picks lowest val_loss
        output_directory=performance_directory_train_val,  # dataset_2su_roi or dataset_3src depending on use_roi
        test_generator=validation_generator2,         # evaluates on training dataset validation split
        model_type=model_type,
        train_type=train_type2,
        fingerprint=fingerprint2,
        timeslices=2,
        soft_quantize_layer=False,
    )
# Outputs: parquet file saved to performance_directory_train_val

### Cleanup

Deletes **Part 2** model and data generators, clears the TF session, and forces garbage collection to free memory before evaluation on the test dataset.

In [ ]:
if not skip_part_2:
    cleanup_models_and_generators([model2, training_generator2, validation_generator2])

## Evaluation on Test Dataset

Evaluates the **Part 2** model on the test dataset (`dataset_2s_roi` when `use_roi=True`, or `dataset_2sc` when `use_roi=False`). Results are saved as parquet files for downstream analysis.

### Generate & Load Test TFRecords

Generates TFRecords for the test dataset using the label scaling from the training dataset, then loads them as a test generator with inputs hard-digitized using the **Part 1** thresholds.

In [ ]:
# generate (or locate) test TFRecords unconditionally — tfrecords_dir_test is needed for evaluation regardless of skip_part_2
_, tfrecords_dir_test = generate_tfrecords(
    dataset_dir=dataset_test_dir,
    train_batch_size=train_batch_size,
    val_batch_size=val_batch_size,
    select_contained=select_contained_test,
    timeslices=timeslices,
    seed=seed,
    max_workers=1,
    tfrecords_exist=tfrecords_exist_test,
    model_type=model_type,
    labels_scale=np.array(labels_scale),   # reuse training dataset scaling for consistent normalization
    test_only=True,                        # only generate test TFRecords, no train/val generators
    load_roi=load_roi_test,
    use_roi_labels=use_roi_labels,
)
print(f"TFRecords dirs test: {tfrecords_dir_test}")

# load test generator unconditionally — test_generator is needed by all evaluation cells below,
# even when skip_part_2=True (i.e. Part 2 training was skipped but evaluation still runs)
test_generator = load_tfrecords(
    tfrecords_dir_test=tfrecords_dir_test,
    noise=noise2,                          # -1: no noise
    quantize=False,
    shuffle=True,
    digitize=True,                         # hard-digitize inputs using Part 1 thresholds
    digitize_levels=initial_levels,        # 2-bit output levels [0, 1, 2, 3]
    digitize_thresholds=thresholds,        # thresholds learned in Part 1
    seed=seed,
    test_only=True,                        # only load test generator, no train/val
)
# Outputs: tfrecords_dir_test, test_generator

### Load Model & Best Checkpoint

> **TODO:** This cell and the code cell below are unused.

In [ ]:
#if not skip_part_2:
#    model3 = create_model(
#        model_type=model_type,
#        timeslices=timeslices,
#        soft_quantize_layer=soft_quantize_layer2,
#    )

### Evaluate on Test Dataset & Save Results

Loads the best **Part 2** checkpoint and runs predictions on the test dataset to evaluate generalization to a different detector geometry.

Results are saved as a parquet file to `performance_dir_test` for downstream analysis.

In [ ]:
# no skip_part_2 guard — evaluation should run regardless of whether Part 2 was trained this session;
# requires checkpoints_directory2 and fingerprint2 to be set (either from training or from the parameters cell)
save_performance_parquet(
    checkpoints=checkpoints_directory2,        # scans all checkpoints, picks lowest val_loss
    output_directory=performance_dir_test,
    test_generator=test_generator,             # test dataset (dataset_2sc or dataset_2su_roi)
    model_type=model_type,
    train_type=train_type2,
    fingerprint=fingerprint2,
    timeslices=timeslices,
    soft_quantize_layer=soft_quantize_layer2,
)
# Outputs: parquet file saved to performance_dir_test

## Visualize Training History

Plots training and validation loss over epochs and saves the figure as a PNG. A red marker indicates the best validation loss epoch.

Run the second cell to zoom in on later epochs if early loss dominates the scale.

In [ ]:
if not skip_part_2:  # history2 only exists when Part 2 training ran this session
    # Plot full training history from epoch 1
    # Saves PNG to: {checkpoints_directory2}/history-training_validation_loss_e1.png
    viz_history(history2, title=model_type, prefix=f'{checkpoints_directory2}/history-')

In [ ]:
if not skip_part_2:  # history2 only exists when Part 2 training ran this session
    # Plot training history from epoch 500 onwards — useful to zoom in on later convergence
    # Saves PNG to: {checkpoints_directory2}/history-training_validation_loss_e500.png
    viz_history(history2, title=model_type, start_epoch=500, prefix=f'{checkpoints_directory2}/history-')

## Model Export & Utilities

Optional cells for loading the best model as an object, saving it in multiple formats, exporting test data as numpy arrays, and running predictions directly.

In [ ]:
if not skip_part_2:  # model2 only exists when Part 2 training ran this session
    # Load the best model checkpoint as a model object (needed for cells below)
    # Note: save_performance_parquet already loads the best model internally for evaluation
    model2_best, bestfile = load_best_model(checkpoints_directory2, model2)
    print(f"Best weights: {bestfile}")

In [ ]:
if not skip_part_2:  # model2_best only exists when Part 2 training ran this session
    # Save the best model in multiple formats (h5, keras, json) to the checkpoints directory
    save_best_model(checkpoints_directory2, model2_best)

In [ ]:
# Optional: reload test_generator from TFRecords if not already in memory
# test_generator is already available from the "Generate & Load Test TFRecords" cell above

# Note: calling OptimizedDataGenerator directly is NOT sufficient here —
# it would skip the digitization step and produce continuous inputs instead of 2-bit.
# Use load_tfrecords with the same parameters as the original cell instead:
#
# from OptimizedDataGenerator_v3 import OptimizedDataGenerator
# test_generator = OptimizedDataGenerator(
#     load_from_tfrecords_dir=tfrecords_dir_test,
#     quantize=False,
# )

test_generator = load_tfrecords(
    tfrecords_dir_test=tfrecords_dir_test,
    noise=noise2,                          # -1: no noise
    quantize=False,
    shuffle=True,
    digitize=True,                         # hard-digitize inputs using Part 1 thresholds
    digitize_levels=initial_levels,        # 2-bit output levels [0, 1, 2, 3]
    digitize_thresholds=thresholds,        # thresholds learned in Part 1
    seed=seed,
    test_only=True,                        # only load test generator, no train/val
)

In [ ]:
if not skip_part_2:  # fingerprint2 only exists when Part 2 training ran this session
    # Save test generator inputs and labels as numpy arrays
    # Filenames are derived from model_type and fingerprint2 to be unique per run
    # ROI-aware: dataset_test_dir reflects the active test dataset (dataset_2sc or dataset_2su_roi)
    save_data_as_npy(
        test_generator,
        x_test_npy=f"npy/{timeslices}t-{model_type}-{fingerprint2}-X_test.npy",
        y_test_npy=f"npy/{timeslices}t-{model_type}-{fingerprint2}-y_test.npy",
    )

In [ ]:
if not skip_part_2:  # model2_best only exists when Part 2 training ran this session
    # Run prediction with the best model on the test generator
    p_test = model2_best.predict(test_generator)

### MSE Evaluation

Computes Mean Squared Error between model predictions and ground truth labels on the test dataset.

In [ ]:
if not skip_part_2:  # p_test only exists when Part 2 training ran this session
    # Output labels per model type
    output_labels = {
        'Max':  ['x', 'y', 'cotA', 'cotB'],
        'Full': ['x', 'y', 'cotA', 'cotB'],
        'Slim': ['x', 'y', 'cotB'],
    }
    key = 'Max' if 'Max' in model_type else 'Full' if 'Full' in model_type else 'Slim'
    labels = output_labels[key]

    # Extract ground truth from test generator
    truth = np.concatenate([y for _, y in test_generator], axis=0)

    # MSE per output and overall
    mse_per_output = np.mean((p_test[:, :len(labels)] - truth[:, :len(labels)])**2, axis=0)
    mse_total = float(np.mean(mse_per_output))

    print(dataset_test_dir)
    print("MSE per output:")
    for label, mse in zip(labels, mse_per_output):
        print(f"  {label:>6}: {mse:.6f}")
    print(f"  {'total':>6}: {mse_total:.6f}")

### Log Run to CSV

Appends this run's configuration and results to `runs.csv` in the repo root, so runs across sessions (e.g. `use_roi=True` vs `use_roi=False`, different `model_type`) can be compared later without re-reading parquets or notebook outputs.

In [ ]:
if not skip_part_2:  # mse_per_output, labels, mse_total only exist when Part 2 evaluation ran this session
    import csv
    from datetime import datetime

    runs_csv = 'runs.csv'
    fieldnames = [
        'timestamp', 'model_type', 'use_roi', 'use_roi_labels',
        'epochs1', 'epochs2', 'seed',
        'fingerprint1', 'best_val_loss_part1', 'thresholds',
        'fingerprint2', 'best_val_loss_part2',
        'dataset_test_dir', 'mse_x', 'mse_y', 'mse_cotA', 'mse_cotB', 'mse_total',
    ]

    row = {
        'timestamp': datetime.now().isoformat(timespec='seconds'),
        'model_type': model_type,
        'use_roi': use_roi,
        'use_roi_labels': use_roi_labels if use_roi else '',
        'epochs1': epochs1,
        'epochs2': epochs2,
        'seed': seed,
        'fingerprint1': fingerprint1 if not skip_part_1 else '',
        'best_val_loss_part1': f"{min(history1.history['val_loss']):.4f}" if not skip_part_1 else '',
        'thresholds': list(np.round(thresholds, 2)),
        'fingerprint2': fingerprint2,
        'best_val_loss_part2': f"{min(history2.history['val_loss']):.4f}",
        'dataset_test_dir': dataset_test_dir,
        'mse_total': f"{mse_total:.6f}",
    }
    for label, mse in zip(labels, mse_per_output):
        row[f'mse_{label}'] = f"{mse:.6f}"

    # DictWriter only writes a header for a new file, so appending after fieldnames has changed
    # would put N values under an N-1 column header and silently misalign every later row.
    # Read the header that is actually on disk and migrate the file if it no longer matches.
    header, existing = None, []
    if os.path.isfile(runs_csv):
        with open(runs_csv, newline='') as f:
            reader = csv.DictReader(f)
            header, existing = reader.fieldnames, list(reader)

    if header is not None and header != fieldnames:
        merged = header + [c for c in fieldnames if c not in header]   # keep existing column order
        with open(runs_csv, 'w', newline='') as f:
            writer = csv.DictWriter(f, fieldnames=merged, restval='')
            writer.writeheader()
            writer.writerows(existing)      # older runs get '' for the new columns
            writer.writerow(row)
        print(f"Logged run to {runs_csv} (schema migrated: {len(header)} -> {len(merged)} columns)")
    else:
        with open(runs_csv, 'a', newline='') as f:
            writer = csv.DictWriter(f, fieldnames=fieldnames, restval='')
            if header is None:
                writer.writeheader()
            writer.writerow(row)
        print(f"Logged run to {runs_csv}")
# Outputs: appends one row to runs.csv

In [ ]:
!cat runs.csv

## Run History

Prints all runs logged so far in `runs.csv` (from the **Log Run to CSV** cell above), formatted as a table — one row per run, across `use_roi`, `model_type`, and other configs.

In [ ]:
import pandas as pd

runs_csv = 'runs.csv'
if os.path.isfile(runs_csv):
    df_runs = pd.read_csv(runs_csv).sort_values('timestamp', ascending=False).reset_index(drop=True)
    with pd.option_context('display.max_columns', None, 'display.width', 200):
        display(df_runs)
else:
    print(f"No {runs_csv} found yet — run the pipeline at least once with skip_part_2=False.")

---
## Residual Diagnostics

Reads the saved performance parquets and plots `residual = true - predicted` in physical units,
for the ROI model and the non-ROI model side by side.

### ROI vs non-ROI

Same architecture and epoch budget for both, so the shapes are directly comparable. `labels_scale`
is read live from each dataset's own `metadata.json`, since it differs between the two.

In [ ]:
# Reads dataset path variables defined in the Parameters cell above
# (dataset_2s_roi_dir, dataset_2sc_dir, performance_directory_2s_roi, performance_directory_2sc).
# These do not depend on use_roi, so they exist however the Parameters cell is configured —
# but that cell must have been run at least once this session.
import os
import json
import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

_model = model_type   # follow the Parameters cell; override here to inspect another model

# Resolve the ROI dataset from roi_dataset, so these plots follow the Parameters cell instead of
# being pinned to one dataset. Independent of use_roi, so they work even after a non-ROI run.
if roi_dataset == '2su':
    _roi_dir, _roi_perf = dataset_2su_roi_dir, (performance_directory_2su_roi if use_roi_labels
                                                else performance_directory_2su_roi_midplane)
elif roi_dataset == '2s':
    _roi_dir, _roi_perf = dataset_2s_roi_dir, (performance_directory_2s_roi if use_roi_labels
                                               else performance_directory_2s_roi_midplane)
else:
    raise ValueError(f"roi_dataset must be '2s' or '2su', got {roi_dataset!r}")
_roi_tfr = 'TFR_test_slim' if use_roi_labels else 'TFR_test_slim_midplane_labels'

def _find_parquet(perf_dir, model):
    # fingerprint depends on the model, not the dataset; glob rather than hardcode it.
    # Returns None when absent: a dataset the current model_type has not been run on is skipped
    # below rather than aborting the whole notebook mid-'Run All'.
    hits = sorted(glob.glob(f'{perf_dir}2t-{model}-2bit_optimized-*-vars.parquet'))
    return hits[-1] if hits else None

def _dataset_display_name(dataset_dir):
    return os.path.basename(dataset_dir.rstrip('/')).removesuffix('_parquets')

def _labels_scale(dataset_dir, tfr_subdir):
    with open(os.path.join(dataset_dir, 'TFR_files', '2t', tfr_subdir, 'metadata.json')) as f:
        return json.load(f)['labels_scale']

_candidates = [
    (f'{_dataset_display_name(_roi_dir)} (ROI)', _roi_perf, _roi_dir, _roi_tfr),
    (_dataset_display_name(dataset_2sc_dir), performance_directory_2sc,
     dataset_2sc_dir, 'TFR_test_contained_slim'),
]

_datasets = []
for _label, _perf, _dsdir, _tfr in _candidates:
    _pq = _find_parquet(_perf, _model)
    if _pq is None:
        print(f"skipping {_label}: no {_model} result in {_perf}")
        continue
    _datasets.append((_label, _pq, _labels_scale(_dsdir, _tfr)))

if not _datasets:
    raise FileNotFoundError(f'no {_model} results found for any dataset - run the pipeline first')

# (pred_col, true_col, column index, colour, panel title, x label, tick spacing) — red/orange/green
# convention borrowed from training_tracker.ipynb; ticks match pixel pitch, 50 um x / 12.5 um y
_cols = [
    ('x',    'xtrue',    0, 'red',    'x residual (\u00b5m)', 'true x - predicted x  (\u00b5m)', 50),
    ('y',    'ytrue',    1, 'orange', 'y residual (\u00b5m)', 'true y - predicted y  (\u00b5m)', 12.5),
    ('cotB', 'cotBtrue', 2, 'green',  'cotBeta residual',  'true cotB - predicted cotB',  None),
]

# pass 1: residuals, counts, and a shared per-column range. The range covers the full extent of
# both datasets (not a percentile), so every event is inside the frame and the two rows stay
# directly comparable; 1.08 padding keeps the min/max markers off the frame.
_res, _cnt, _xlim = {}, {}, {}
for r, (lab, path, sc) in enumerate(_datasets):
    df_investigate = pd.read_parquet(path); _cnt[r] = len(df_investigate)
    for pc, tc, ci, *_ in _cols:
        v = (df_investigate[tc] - df_investigate[pc]) * sc[ci]
        _res[(r, ci)] = v
        _xlim[ci] = max(_xlim.get(ci, 0), np.abs(v).max())

print(f"{'run':>34} {'x std':>9} {'y std':>9} {'cotB std':>10}   (std = spread about the mean; bias shown on plot)")
for r, (lab, _, _) in enumerate(_datasets):
    print(f"{lab:>34} {_res[(r,0)].std():>8.2f}\u00b5m {_res[(r,1)].std():>8.2f}\u00b5m {_res[(r,2)].std():>10.4f}")

# squeeze=False keeps axes 2-D even with a single dataset, so axes[r, ci] stays valid
fig, axes = plt.subplots(len(_datasets), 3, figsize=(18, 4.6 * len(_datasets)), squeeze=False)
for r, (lab, _, _) in enumerate(_datasets):
    for pc, tc, ci, color, title, xlabel, tick in _cols:
        ax = axes[r, ci]; v = _res[(r, ci)]; lim = _xlim[ci] * 1.08
        ax.hist(v, bins=250, range=(-lim, lim), color=color, edgecolor='none')
        zero_line = ax.axvline(0, color='black', ls='--', lw=1)
        # this panel's own extremes - dark gray, dotted, no legend entry
        ax.axvline(v.min(), color='dimgray', ls=':', lw=1.5)
        ax.axvline(v.max(), color='dimgray', ls=':', lw=1.5)
        ax.set_xlim(-lim, lim)
        ax.set_xlabel(xlabel); ax.set_ylabel('Number of test events')
        ax.set_title(f'{title}\n{lab}  (N={_cnt[r]:,})', fontsize=10)
        ax.text(0.02, 0.95, f'std  {v.std():.2f}\nbias {v.mean():+.2f}', transform=ax.transAxes,
                va='top', fontsize=9, family='monospace',
                bbox=dict(fc='white', ec='0.7', alpha=0.85))
        if tick is not None:
            ax.xaxis.set_major_locator(mticker.MultipleLocator(tick))
            ax.tick_params(axis='x', labelrotation=45)

fig.suptitle('Residuals: true minus predicted', y=1.005)
fig.legend([zero_line], ['perfect prediction (residual = 0)'], loc='lower center', bbox_to_anchor=(0.5, -0.015))
fig.tight_layout()
fig.show()

### ROI only

Same plot as above, but just the ROI row, at a larger size for a closer look at the x bimodality.

In [ ]:
# Same conventions as the ROI-vs-non-ROI cell above, but ROI only (single dataset, no shared xlim needed).
import os
import json
import glob
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

_model = model_type   # follow the Parameters cell; override here to inspect another model

# Resolve the ROI dataset from roi_dataset, so these plots follow the Parameters cell instead of
# being pinned to one dataset. Independent of use_roi, so they work even after a non-ROI run.
if roi_dataset == '2su':
    _roi_dir, _roi_perf = dataset_2su_roi_dir, (performance_directory_2su_roi if use_roi_labels
                                                else performance_directory_2su_roi_midplane)
elif roi_dataset == '2s':
    _roi_dir, _roi_perf = dataset_2s_roi_dir, (performance_directory_2s_roi if use_roi_labels
                                               else performance_directory_2s_roi_midplane)
else:
    raise ValueError(f"roi_dataset must be '2s' or '2su', got {roi_dataset!r}")
_roi_tfr = 'TFR_test_slim' if use_roi_labels else 'TFR_test_slim_midplane_labels'

def _find_parquet(perf_dir, model):
    # fingerprint depends on the model, not the dataset; glob rather than hardcode it
    hits = sorted(glob.glob(f'{perf_dir}2t-{model}-2bit_optimized-*-vars.parquet'))
    if not hits:
        raise FileNotFoundError(f'no {model} result in {perf_dir} - run the pipeline first')
    return hits[-1]

def _dataset_display_name(dataset_dir):
    return os.path.basename(dataset_dir.rstrip('/')).removesuffix('_parquets')

def _labels_scale(dataset_dir, tfr_subdir):
    with open(os.path.join(dataset_dir, 'TFR_files', '2t', tfr_subdir, 'metadata.json')) as f:
        return json.load(f)['labels_scale']

_roi_label = f'{_dataset_display_name(_roi_dir)} (ROI)'
_roi_parquet = _find_parquet(_roi_perf, _model)
_roi_scales = _labels_scale(_roi_dir, _roi_tfr)

_cols = [
    ('x', 'xtrue', 0, 'red', 'x residual (µm)', 'true x - predicted x  (µm)', 50),
    ('y', 'ytrue', 1, 'orange', 'y residual (µm)', 'true y - predicted y  (µm)', 12.5),
    ('cotB', 'cotBtrue', 2, 'green', 'cotBeta residual', 'true cotB - predicted cotB', None),
]

df_roi_only = pd.read_parquet(_roi_parquet)
print(f"{_roi_label}: {len(df_roi_only):,} events")

fig, axes = plt.subplots(1, 3, figsize=(18, 6))
for pred_col, true_col, col_idx, color, title, xlabel, tick_spacing in _cols:
    ax = axes[col_idx]
    residual = (df_roi_only[true_col] - df_roi_only[pred_col]) * _roi_scales[col_idx]
    ax.hist(residual, bins=150, color=color, edgecolor='none')
    zero_line = ax.axvline(0, color='black', ls='--', lw=1)
    ax.axvline(residual.min(), color='dimgray', ls=':', lw=1.5)
    ax.axvline(residual.max(), color='dimgray', ls=':', lw=1.5)
    ax.set_xlabel(xlabel)
    ax.set_ylabel('Number of test events')
    ax.set_title(title)
    xlim = residual.abs().max() * 1.08
    ax.set_xlim(-xlim, xlim)
    if tick_spacing is not None:
        ax.xaxis.set_major_locator(mticker.MultipleLocator(tick_spacing))
        ax.tick_params(axis='x', labelrotation=45)

fig.suptitle(f'Residuals: True minus Predicted — {_roi_label}  (N={len(df_roi_only):,})', y=1.03)
fig.legend([zero_line], ['perfect prediction (residual = 0)'], loc='lower center', bbox_to_anchor=(0.5, -0.05), ncol=1)
fig.tight_layout()
fig.show()